# Land Cover & Development — Goat Island
Run locally from the repository root or `notebooks` folder. This notebook is independent of the other notebooks and uses the seven project-confirmed parcels selected by **Parcel_Num** in `Park_Parcels/Park_Parcles.shp`.

**Questions:** What land cover exists inside the selected parcel? How does it differ from the surrounding one-mile ring? Where are impervious surfaces concentrated?

Outputs go to `goat_island_outputs/land_cover/`: tables, PNG figures, GeoTIFFs, study zones, and a run manifest. The selected parcel is approximately 486.6 acres after dissolving all seven specified parcels.

Sources: [Annual NLCD / USGS](https://www.usgs.gov/centers/eros/science/nlcd-product-suite), [community-hosted Earth Engine mirror](https://gee-community-catalog.org/projects/annual_nlcd/), [NAIP](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ). The mirror is separate from the official USGS distribution; image IDs and metadata are recorded at runtime. No results are precomputed in this notebook.

Run cells in order. Settings and helpers are defined once; analytical summaries, raster exports, maps, and optional aerial review follow separately. Figures are PNG only.

Install dependencies once if needed: `earthengine-api geemap geopandas shapely pyogrio rasterio matplotlib numpy pandas requests pillow`.

## 1. Imports

Load only the libraries used here.

In [ ]:
import json
import warnings
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from PIL import Image
from rasterio.features import geometry_mask
from rasterio.plot import show
from shapely import force_2d


## 2. Settings, classes, and local paths

Keep the analysis year, optional outputs, parcel IDs, data sources, class labels, and colors together.

In [ ]:
EE_PROJECT = "stalwart-glider-279103"
PREFERRED_YEAR = 2024
ALLOW_OLDER_YEAR = False
RUN_NAIP = True
EXPORT_RASTERS = True
METRIC_CRS = "EPSG:26914"
ACRE_M2 = 4046.8564224
PROJECT_ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "Park_Parcels/Park_Parcles.shp").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the goat-island-analysis folder.")
OUTPUT_DIR = PROJECT_ROOT / "goat_island_outputs" / "land_cover"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GOAT_ISLAND_PARCEL_IDS = [
    "65079661510030000",
    "65130905010020000",
    "65130905010030000",
    "65130905010050000",
    "65130905010060000",
    "65146334010100000",
    "65146334010140000",
]
LC_ASSET = "projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/LANDCOVER"
IMP_ASSET = (
    "projects/sat-io/open-datasets/USGS/ANNUAL_NLCD/FRACTIONAL_IMPERVIOUS_SURFACE"
)
CLASSES = {
    11: ("Open water", "#466b9f"),
    12: ("Perennial ice/snow", "#d1def8"),
    21: ("Developed, open space", "#dec5c5"),
    22: ("Developed, low intensity", "#d99282"),
    23: ("Developed, medium intensity", "#eb0000"),
    24: ("Developed, high intensity", "#ab0000"),
    31: ("Barren land", "#b3ac9f"),
    41: ("Deciduous forest", "#68ab5f"),
    42: ("Evergreen forest", "#1c5f2c"),
    43: ("Mixed forest", "#b5c58f"),
    52: ("Shrub/scrub", "#ccb879"),
    71: ("Grassland/herbaceous", "#dfdfc2"),
    81: ("Pasture/hay", "#dcd939"),
    82: ("Cultivated crops", "#ab6c28"),
    90: ("Woody wetlands", "#b8d9eb"),
    95: ("Emergent herbaceous wetlands", "#6c9fb8"),
}
codes = list(CLASSES)


## 3. Earth Engine connection

Reuse authentication when available.

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready. Outputs:", OUTPUT_DIR)


## 4. Reusable helpers

Shared geometry, grid, export, and map rules are defined once.

In [ ]:
def to_ee(geom):
    """Convert a projected local geometry into a two-dimensional Earth Engine geometry."""
    wgs = gpd.GeoSeries([geom], crs=METRIC_CRS).to_crs(4326).iloc[0]
    return ee.Geometry(json.loads(json.dumps(wgs.__geo_interface__)))


def available_years(collection):
    """Read the years explicitly advertised by the collection metadata."""
    years = collection.aggregate_array("year").getInfo()
    if not years:
        raise RuntimeError(
            "The collection has no year metadata; inspect its schema before proceeding."
        )
    return sorted({int(y) for y in years})


def select_year(collection):
    """Require one image for the selected year; never silently mosaic ambiguous results."""
    subset = collection.filter(ee.Filter.eq("year", YEAR))
    if subset.size().getInfo() != 1:
        raise RuntimeError(
            "Expected one national image per year. Inspect the collection before mosaicking."
        )
    return ee.Image(subset.first())


def grid(projection):
    """Use the product native grid for area statistics."""
    return {
        "crs": projection["crs"],
        "crsTransform": projection["transform"],
        "maxPixels": 100000000,
        "tileScale": 4,
    }


def save_manifest():
    """Record the selected inputs, grid, and optional imagery provenance."""
    (OUTPUT_DIR / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )


def save_figure(fig, name):
    """Save a PNG figure and display it in the notebook."""
    fig.savefig(OUTPUT_DIR / f"{name}_{YEAR}.png", dpi=220, bbox_inches="tight")
    plt.show()


def decorate_map(ax, crs):
    """Apply consistent boundaries, coordinate labels, orientation, and scale bar."""
    projected = zones.to_crs(crs)
    projected.iloc[[0]].boundary.plot(ax=ax, color="black", linewidth=1.5)
    outer = gpd.GeoSeries([buffer], crs=METRIC_CRS).to_crs(crs)
    outer.boundary.plot(ax=ax, color="black", linestyle="--", linewidth=1)
    ax.set(xlabel="Easting (m)", ylabel="Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
    ax.annotate(
        "N",
        xy=(0.95, 0.94),
        xytext=(0.95, 0.84),
        xycoords="axes fraction",
        ha="center",
        arrowprops={"arrowstyle": "->", "color": "black"},
    )
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x = xmin + 0.06 * (xmax - xmin)
    y = ymin + 0.06 * (ymax - ymin)
    ax.plot([x, x + 1000], [y, y], color="black", linewidth=3)
    ax.text(x + 500, y + 0.015 * (ymax - ymin), "1 km", ha="center", fontsize=9)


def review_aerial_imagery():
    """Review dated NAIP imagery and interactive layers; retain core outputs if imagery fails."""
    try:
        naip_all = (
            ee.ImageCollection("USDA/NAIP/DOQQ")
            .filterBounds(study_region)
            .filterDate("2010-01-01", datetime.now(timezone.utc).strftime("%Y-%m-%d"))
        )
        if naip_all.size().getInfo() == 0:
            raise RuntimeError("No NAIP imagery found.")
        latest = ee.Image(naip_all.sort("system:time_start", False).first())
        naip_year = int(
            ee.Date(latest.get("system:time_start")).format("YYYY").getInfo()
        )
        tiles = naip_all.filterDate(f"{naip_year}-01-01", f"{naip_year + 1}-01-01")
        dates = sorted(
            {
                datetime.fromtimestamp(t / 1000, timezone.utc).date().isoformat()
                for t in tiles.aggregate_array("system:time_start").getInfo()
            }
        )
        naip = tiles.sort("system:time_start").mosaic().select(["R", "G", "B"])
        manifest["naip"] = {
            "year": naip_year,
            "acquisition_dates": dates,
            "image_ids": tiles.aggregate_array("system:index").getInfo(),
        }
        save_manifest()
        print("NAIP acquisition dates:", dates)
        preview_geom = preserve.buffer(150).envelope
        xmin, ymin, xmax, ymax = preview_geom.bounds
        preview_region = ee.Geometry.Rectangle(
            [xmin, ymin, xmax, ymax], proj=METRIC_CRS, geodesic=False
        )
        common = {
            "region": preview_region,
            "crs": METRIC_CRS,
            "dimensions": 1100,
            "format": "png",
        }
        lc_vis = lc.remap(codes, list(range(len(codes)))).visualize(
            min=0, max=len(codes) - 1, palette=[CLASSES[c][1] for c in codes]
        )
        rgb_vis = naip.visualize(bands=["R", "G", "B"], min=0, max=255)
        fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
        for ax, img, label in zip(
            axes,
            [lc_vis, rgb_vis],
            [f"Annual NLCD {YEAR} — 30 m", f"NAIP {naip_year} — aerial reference"],
        ):
            response = requests.get(img.getThumbURL(common), timeout=120)
            response.raise_for_status()
            ax.imshow(
                Image.open(BytesIO(response.content)),
                extent=[xmin, xmax, ymin, ymax],
                interpolation="nearest",
            )
            gpd.GeoSeries([preserve], crs=METRIC_CRS).boundary.plot(
                ax=ax, color="red", linewidth=1
            )
            ax.set(title=label, xlabel="UTM Easting (m)", ylabel="UTM Northing (m)")
            ax.ticklabel_format(style="plain", useOffset=False)
        fig.suptitle("Goat Island parcel — land-cover consistency review")
        fig.text(
            0.5,
            0.01,
            "NAIP dates: "
            + ", ".join(dates)
            + " • red: selected parcel • imagery dates differ from NLCD where indicated",
            ha="center",
            fontsize=8,
        )
        fig.tight_layout(rect=[0, 0.07, 1, 0.92])
        save_figure(fig, "aerial_comparison")
        m = geemap.Map()
        m.centerObject(study_region, 13)
        m.addLayer(
            naip, {"bands": ["R", "G", "B"], "min": 0, "max": 255}, f"NAIP {naip_year}"
        )
        m.addLayer(lc_vis, {}, f"NLCD {YEAR}", False)
        m.addLayer(
            imp,
            {"min": 0, "max": 100, "palette": ["ffffff", "fdae61", "a50026"]},
            f"Impervious {YEAR}",
            False,
        )
        m.addLayer(
            ee.FeatureCollection([ee.Feature(regions["Selected parcel"])]).style(
                color="ff0000", fillColor="00000000", width=2
            ),
            {},
            "Selected parcel",
        )
        m.add_legend(
            title=f"Annual NLCD {YEAR}",
            legend_dict={CLASSES[c][0]: CLASSES[c][1] for c in present},
        )
        display(m)
    except Exception as exc:
        warnings.warn(
            f"NAIP review did not finish: {exc}. Core NLCD outputs remain available; fix and rerun this cell."
        )
        manifest["naip_review_error"] = str(exc)
        save_manifest()


## 5. Parcel boundary and surrounding ring

Keep the selected seven-parcel extent separate from the one-mile ring outside it.

In [ ]:
parcels = gpd.read_file(PROJECT_ROOT / "Park_Parcels/Park_Parcles.shp")
if parcels.crs is None:
    raise ValueError("Parcel CRS is missing.")
parcel_numbers = parcels["Parcel_Num"].astype(str).str.strip()
missing_ids = set(GOAT_ISLAND_PARCEL_IDS) - set(parcel_numbers)
if missing_ids:
    raise ValueError(f"Required Goat Island parcels are missing: {sorted(missing_ids)}")
selected = parcels.loc[parcel_numbers.isin(GOAT_ISLAND_PARCEL_IDS)].copy()
if selected.empty or selected.geometry.isna().any() or selected.geometry.is_empty.any():
    raise ValueError("Goat Island parcels are missing or have empty geometry.")
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if not selected.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Expected polygon parcels.")
preserve = selected.dissolve().to_crs(METRIC_CRS).geometry.iloc[0]
buffer = preserve.buffer(1609.344)
ring = buffer.difference(preserve)
zones = gpd.GeoDataFrame(
    {"zone": ["Selected parcel", "Surrounding one-mile ring"]},
    geometry=[preserve, ring],
    crs=METRIC_CRS,
)
zones["geometry_acres"] = zones.area / ACRE_M2
zones.to_crs(4326).to_file(OUTPUT_DIR / "study_zones.geojson", driver="GeoJSON")
regions = {row.zone: to_ee(row.geometry) for row in zones.itertuples()}
study_region = to_ee(buffer)
display(zones.drop(columns="geometry"))
assert preserve.intersection(ring).area < 0.01


## 6. Select and mask matching products

Require the same year for land cover and impervious surface. Preserve native grids and valid zero imperviousness.

In [ ]:
lc_collection = ee.ImageCollection(LC_ASSET)
imp_collection = ee.ImageCollection(IMP_ASSET)
common = sorted(
    set(available_years(lc_collection)) & set(available_years(imp_collection))
)
if PREFERRED_YEAR in common:
    YEAR = PREFERRED_YEAR
elif ALLOW_OLDER_YEAR and any((y < PREFERRED_YEAR for y in common)):
    YEAR = max((y for y in common if y < PREFERRED_YEAR))
    warnings.warn(f"Using explicitly permitted earlier year: {YEAR}")
else:
    raise RuntimeError(
        f"Requested {PREFERRED_YEAR}; common available years: {common}. Change the year or explicitly allow an earlier year."
    )
lc_raw, imp_raw = (select_year(lc_collection), select_year(imp_collection))
lc = lc_raw.select([0]).rename("land_cover")
lc = lc.updateMask(lc.remap(codes, [1] * len(codes), 0).eq(1))
imp = imp_raw.select([0]).rename("impervious_pct")
imp = imp.updateMask(imp.gte(0).And(imp.lte(100)))
lc_projection = lc.projection().getInfo()
imp_projection = imp.projection().getInfo()
for projection in [lc_projection, imp_projection]:
    if "crs" not in projection:
        projection["crs"] = projection["wkt"]


## 7. Record product provenance

Save actual image metadata, product versions, grid definitions, and parcel IDs.

In [ ]:
manifest = {
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "year": YEAR,
    "boundary_source": "Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values",
    "parcel_ids": selected.Parcel_Num.astype(str).tolist(),
    "land_cover": {
        "asset": LC_ASSET,
        "metadata": lc_raw.toDictionary().getInfo(),
        "projection": lc_projection,
    },
    "impervious": {
        "asset": IMP_ASSET,
        "metadata": imp_raw.toDictionary().getInfo(),
        "projection": imp_projection,
    },
    "area_method": "pixelArea sums on each product native grid; weighted boundary pixels; percentages use valid raster area",
}
save_manifest()
print("Analysis year:", YEAR)
print("Land-cover bands:", lc_raw.bandNames().getInfo())
print("Impervious bands:", imp_raw.bandNames().getInfo())


## 8. Calculate land cover, imperviousness, and coverage

Compute native-grid areas by zone. Impervious area weights each cell by its impervious fraction; NoData is excluded.

In [ ]:
records, coverage, impervious_records = ([], [], [])
for zone, region in regions.items():
    geom_acres = float(zones.loc[zones.zone.eq(zone), "geometry_acres"].iloc[0])
    grouped = (
        ee.Image.pixelArea()
        .rename("area")
        .addBands(lc)
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
            geometry=region,
            **grid(lc_projection),
        )
        .getInfo()
        .get("groups", [])
    )
    areas = {int(x["class"]): x["sum"] / ACRE_M2 for x in grouped}
    valid_acres = sum(areas.values())
    if valid_acres <= 0:
        raise RuntimeError(f"No classified land-cover pixels in {zone}.")
    for code, (label, color) in CLASSES.items():
        acres = areas.get(code, 0)
        records.append(
            {
                "zone": zone,
                "year": YEAR,
                "class_code": code,
                "class_name": label,
                "area_acres": acres,
                "pct_valid_area": 100 * acres / valid_acres,
            }
        )
    valid_area = ee.Image.pixelArea().updateMask(imp.mask()).rename("valid_area")
    weighted = imp.divide(100).multiply(ee.Image.pixelArea()).rename("impervious_area")
    sums = (
        valid_area.addBands(weighted)
        .reduceRegion(ee.Reducer.sum(), region, **grid(imp_projection))
        .getInfo()
    )
    if not sums.get("valid_area"):
        raise RuntimeError(f"No valid impervious pixels in {zone}.")
    imp_acres = sums["valid_area"] / ACRE_M2
    impervious_records.append(
        {
            "zone": zone,
            "year": YEAR,
            "valid_area_acres": imp_acres,
            "impervious_area_acres": sums["impervious_area"] / ACRE_M2,
            "mean_impervious_pct": 100 * sums["impervious_area"] / sums["valid_area"],
        }
    )
    for product, measured in [("land_cover", valid_acres), ("impervious", imp_acres)]:
        ratio = measured / geom_acres
        coverage.append(
            {
                "zone": zone,
                "product": product,
                "geometry_acres": geom_acres,
                "valid_raster_acres": measured,
                "raster_to_geometry_pct": 100 * ratio,
            }
        )
        if not 0.98 <= ratio <= 1.02:
            warnings.warn(
                f"Review {product} coverage in {zone}: {ratio:.1%} of vector area."
            )


## 9. Save and review summary tables

Keep percentages and coverage denominators explicit.

In [ ]:
land_cover_df = pd.DataFrame(records)
impervious_df = pd.DataFrame(impervious_records)
coverage_df = pd.DataFrame(coverage)
for name, frame in [
    ("land_cover_summary", land_cover_df),
    ("impervious_summary", impervious_df),
    ("coverage_checks", coverage_df),
]:
    frame.to_csv(OUTPUT_DIR / f"{name}_{YEAR}.csv", index=False)
display(coverage_df.round(2))
display(land_cover_df.loc[land_cover_df.area_acres.gt(0)].round(2))
display(impervious_df.round(2))


## 10. Compare land-cover composition

Use the same class order in the parcel and surrounding-ring charts.

In [ ]:
present = [
    c
    for c in codes
    if land_cover_df.loc[land_cover_df.class_code.eq(c), "area_acres"].sum() > 0
]
fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(present) * 0.48)), sharey=True)
for ax, zone in zip(axes, regions):
    table = (
        land_cover_df[land_cover_df.zone.eq(zone)].set_index("class_code").loc[present]
    )
    ax.barh(
        range(len(present)),
        table.pct_valid_area,
        color=[CLASSES[c][1] for c in present],
        edgecolor="#555555",
        linewidth=0.4,
    )
    ax.set_yticks(range(len(present)), [CLASSES[c][0] for c in present])
    for y, row in enumerate(table.itertuples()):
        ax.text(
            row.pct_valid_area + 0.8,
            y,
            f"{row.pct_valid_area:.1f}% | {row.area_acres:.1f} ac",
            va="center",
            fontsize=8,
        )
    ax.set_xlim(0, 120)
    ax.set_xlabel("Valid classified area (%)")
    ax.set_title(zone)
    ax.grid(axis="x", alpha=0.2)
axes[0].invert_yaxis()
fig.suptitle(f"Goat Island land cover — Annual NLCD {YEAR}")
fig.text(
    0.5,
    -0.02,
    "Source: USGS Annual NLCD via community Earth Engine mirror • 30 m • selected parcel and surrounding ring",
    ha="center",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "land_cover_comparison")


## 11. Compare impervious cover

Show area-weighted impervious percentage and equivalent impervious acres.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(impervious_df.zone, impervious_df.mean_impervious_pct, color="#9d453b")
for bar, row in zip(bars, impervious_df.itertuples()):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{row.mean_impervious_pct:.1f}% | {row.impervious_area_acres:.1f} ac",
        va="center",
    )
ax.set(
    xlim=(0, 115),
    xlabel="Area-weighted impervious surface (%)",
    title=f"Goat Island development context — {YEAR}",
)
fig.text(
    0.5,
    -0.02,
    "Source: USGS Annual NLCD fractional impervious surface • 30 m",
    ha="center",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "impervious_comparison")


## 12. Export GIS rasters

Keep the native grid, preserve valid zero values, and explicitly mark areas outside the study boundary as NoData.

In [ ]:
lc_path = OUTPUT_DIR / f"land_cover_{YEAR}.tif"
imp_path = OUTPUT_DIR / f"impervious_{YEAR}.tif"
if EXPORT_RASTERS:
    for image, path, projection in [
        (lc, lc_path, lc_projection),
        (imp, imp_path, imp_projection),
    ]:
        if path.exists():
            path.unlink()
        geemap.ee_export_image(
            image.clip(study_region).unmask(255),
            filename=str(path),
            region=study_region.bounds(),
            crs=projection["crs"],
            crs_transform=projection["transform"],
            file_per_band=False,
            timeout=180,
        )
        if not path.exists():
            raise RuntimeError(
                f"Raster export failed: {path}. Check Earth Engine access and rerun."
            )
        with rasterio.open(path, "r+") as exported:
            outline = (
                gpd.GeoSeries([buffer], crs=METRIC_CRS).to_crs(exported.crs).iloc[0]
            )
            inside = geometry_mask(
                [outline.__geo_interface__],
                out_shape=(exported.height, exported.width),
                transform=exported.transform,
                invert=True,
            )
            data = exported.read(1)
            data[~inside] = 255
            exported.write(data, 1)
            exported.nodata = 255


## 13. Draw land-cover and impervious maps

Use the existing class palette and a continuous 0–100% impervious scale.

In [ ]:
if lc_path.exists() and imp_path.exists():
    cmap = ListedColormap([CLASSES[c][1] for c in codes])
    cmap.set_bad("#eeeeee")
    fig, ax = plt.subplots(figsize=(11, 9))
    with rasterio.open(lc_path) as src:
        values = src.read(1)
        indexed = np.full(values.shape, np.nan)
        for i, c in enumerate(codes):
            indexed[values == c] = i
        show(
            np.ma.masked_invalid(indexed),
            transform=src.transform,
            ax=ax,
            cmap=cmap,
            norm=BoundaryNorm(np.arange(-0.5, len(codes) + 0.5), len(codes)),
            interpolation="nearest",
        )
        decorate_map(ax, src.crs)
    legend = [Patch(facecolor=CLASSES[c][1], label=CLASSES[c][0]) for c in present]
    ax.legend(
        handles=legend,
        loc="upper left",
        bbox_to_anchor=(1.01, 1),
        title="NLCD class",
        fontsize=8,
    )
    ax.set_title(f"Goat Island land cover and one-mile surroundings — {YEAR}")
    fig.text(
        0.5,
        0.01,
        "USGS Annual NLCD • 30 m • solid: selected parcel; dashed: one-mile boundary; gray: no valid data",
        ha="center",
        fontsize=8,
    )
    save_figure(fig, "land_cover_map")
    fig, ax = plt.subplots(figsize=(10, 9))
    imp_cmap = plt.get_cmap("magma_r").copy()
    imp_cmap.set_bad("#eeeeee")
    with rasterio.open(imp_path) as src:
        values = src.read(1).astype(float)
        values[(values < 0) | (values > 100)] = np.nan
        show(
            np.ma.masked_invalid(values),
            transform=src.transform,
            ax=ax,
            cmap=imp_cmap,
            vmin=0,
            vmax=100,
            interpolation="nearest",
        )
        decorate_map(ax, src.crs)
    fig.colorbar(
        ax.images[0], ax=ax, shrink=0.7, label="Impervious surface per 30 m cell (%)"
    )
    ax.set_title(f"Goat Island impervious surface — {YEAR}")
    fig.text(
        0.5,
        0.01,
        "USGS Annual NLCD • solid: selected parcel; dashed: one-mile boundary",
        ha="center",
        fontsize=9,
    )
    save_figure(fig, "impervious_map")
else:
    print("Raster maps skipped. Enable EXPORT_RASTERS and rerun to generate them.")


## 14. Optional aerial and interactive review

Use RUN_NAIP to control the dated imagery comparison. Aerial review supports interpretation; it is not a formal accuracy assessment.

In [ ]:
if RUN_NAIP:
    review_aerial_imagery()
else:
    print("Aerial imagery review skipped by configuration.")


## 7. Review before integrating into suitability
- Confirm the selected parcel extent matches the intended project scope.
- Check coverage tables and aerial panels; note imagery-year differences and classification disagreements.
- Keep wetlands, forests, and open water distinct. NLCD categories are not habitat-quality measurements or legal wetland determinations.
- Treat developed open space separately from paved surfaces. Imperviousness measures surface cover, not observed runoff, pollution, or road safety.
- Retain 30 m resolution for analysis; use nearest-neighbor resampling for class maps. Future overlays need explicit alignment to a common grid.
- Use these reviewed layers alongside verified entrances, network accessibility, flood hazards, and terrain. This notebook does not produce the final integrated access suitability score.

The aerial comparison requires human review. Do not report a classification accuracy percentage without a designed independent reference sample.

## 15. Output inventory

Review the generated maps, tables, and coverage checks before sharing.

In [ ]:
print("Generated files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)
print("Review the PNG maps and coverage checks before sharing results.")
